# 07 - SageMaker Pipeline (CI/CD DAG)

End-to-end ML pipeline that orchestrates training → evaluation → conditional
registration. The pipeline can finish in two terminal states depending on
the `MinF1Threshold` parameter: success (RegisterModel) or failure (FailStep).

Pipeline structure:
```
TrainRandomForest  →  EvaluateModel  →  CheckF1Threshold
                                              │
                                ┌─────────────┴─────────────┐
                                ↓ (F1 ≥ threshold)          ↓ (F1 < threshold)
                          RegisterModel             ModelQualityBelowThreshold
                                                            (FailStep)
```

**Prerequisites:** Run notebooks 01–06 first. This notebook depends on:
- `scripts/train_sentiment.py` (written by notebook 06 Section 5)
- splits in `s3://<bucket>/splits/` (written by notebook 04)
- Model Package Group `yelp-sentiment-model-group` (created by notebook 06 Section 9)

## 0. Install Dependencies

In [16]:
import importlib
import subprocess
import sys


def is_importable(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except ImportError:
        return False


needs_install = []
if not is_importable("sagemaker.workflow.pipeline"):
    needs_install.append("sagemaker>=2,<3")

if needs_install:
    print(f"Installing: {needs_install}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *needs_install],
        check=True,
    )
    print("\nNew packages installed. Restarting kernel...")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception as restart_error:
        print(f"Auto-restart failed ({restart_error}). Restart manually.")
else:
    print("All dependencies already present. No restart needed.")

All dependencies already present. No restart needed.


## 1. Setup

Loads config from `project_config.json` and auto-detects IAM role.

In [17]:
import json
from pathlib import Path
from time import gmtime, strftime

import boto3
import pandas as pd

import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.steps import TrainingStep, ProcessingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.properties import PropertyFile

default_config = {
    "REGION": "us-east-2",
    "SOURCE_BUCKET": "aai-540-group1-yelp-reviews",
    "FEATURE_COLS": [
        "review_length", "word_count", "useful", "funny", "cool", "vader_score"
    ],
    "TARGET_COL": "sentiment",
    "RANDOM_STATE": 42,
}

config_path = Path("project_config.json")
if config_path.exists():
    with config_path.open() as f:
        cfg = json.load(f)
else:
    cfg = default_config

REGION = cfg.get("REGION", default_config["REGION"])
SOURCE_BUCKET = cfg.get("SOURCE_BUCKET", default_config["SOURCE_BUCKET"])
FEATURE_COLS = cfg.get("FEATURE_COLS", default_config["FEATURE_COLS"])
TARGET_COL = cfg.get("TARGET_COL", default_config["TARGET_COL"])
RANDOM_STATE = cfg.get("RANDOM_STATE", default_config["RANDOM_STATE"])

TAGS = [
    {"Key": "Project", "Value": "yelp-sentiment"},
    {"Key": "Module", "Value": "AAI-540"},
]

session = boto3.Session(region_name=REGION)
s3 = session.client("s3")
sagemaker_client = session.client("sagemaker")
sm_session = sagemaker.Session(boto_session=session)


def get_role():
    try:
        return sagemaker.get_execution_role()
    except Exception:
        pass
    try:
        with open("/opt/ml/metadata/resource-metadata.json") as f:
            return json.load(f)["ExecutionRoleArn"]
    except Exception:
        pass
    try:
        return session.client("iam").get_role(RoleName="LabRole")["Role"]["Arn"]
    except Exception:
        pass
    raise RuntimeError("Could not detect IAM role. Set MANUAL_ROLE_ARN.")


MANUAL_ROLE_ARN = ""
role = MANUAL_ROLE_ARN if MANUAL_ROLE_ARN else get_role()

project_prefix = "sentiment-pipeline"
run_id = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

# Sanity check: training script from notebook 06 must exist AND must have the
# sample-seed argument we depend on for scheduled-run variation.
training_script_path = Path("scripts/train_sentiment.py")
assert training_script_path.exists(), (
    f"Missing {training_script_path}. Run notebook 06 Section 5 first to generate it."
)
script_text = training_script_path.read_text()
assert "--sample-seed" in script_text, (
    "scripts/train_sentiment.py is missing --sample-seed support. "
    "Re-run notebook 06 cell `518cf17c` to regenerate the latest version."
)

print("Region:", REGION)
print("Bucket:", SOURCE_BUCKET)
print("Role:", role)
print("Run ID:", run_id)

Region: us-east-1
Bucket: aai540-group1-yelp-data
Role: arn:aws:iam::476629097825:role/LabRole
Run ID: 2026-05-29-03-50-11


## 2. Upload Training Inputs to S3

Pipelines need CSV inputs at well-known S3 paths. We re-upload splits here
(cheap, ~MB) so this notebook is independent of notebook 06's run state.

In [18]:
training_input_prefix = f"{project_prefix}/training-input/{run_id}"


def upload_split_csv(split_name):
    local_parquet = f"/tmp/{split_name}.parquet"
    local_csv = f"/tmp/{split_name}.csv"
    s3.download_file(SOURCE_BUCKET, f"splits/{split_name}.parquet", local_parquet)
    df = pd.read_parquet(local_parquet)
    df[FEATURE_COLS + [TARGET_COL]].to_csv(local_csv, index=False)
    s3_key = f"{training_input_prefix}/{split_name}.csv"
    s3.upload_file(local_csv, SOURCE_BUCKET, s3_key)
    return f"s3://{SOURCE_BUCKET}/{s3_key}"


train_csv_s3 = upload_split_csv("train")
validation_csv_s3 = upload_split_csv("validation")
test_csv_s3 = upload_split_csv("test")

print("Train CSV:", train_csv_s3)
print("Validation CSV:", validation_csv_s3)
print("Test CSV:", test_csv_s3)

Train CSV: s3://aai540-group1-yelp-data/sentiment-pipeline/training-input/2026-05-29-03-50-11/train.csv
Validation CSV: s3://aai540-group1-yelp-data/sentiment-pipeline/training-input/2026-05-29-03-50-11/validation.csv
Test CSV: s3://aai540-group1-yelp-data/sentiment-pipeline/training-input/2026-05-29-03-50-11/test.csv


## 3. Evaluation Script

A small standalone script the EvaluateModel step runs inside an SKLearn
container. Loads the trained model + test split, computes metrics, writes
`evaluation.json` — which becomes a PropertyFile feeding the ConditionStep.

Uses the same SKLearn 1.2-1 framework as training to avoid pickle-version
skew when loading the model.

In [19]:
import os

os.makedirs("scripts", exist_ok=True)

evaluation_script = r'''
import argparse
import json
import os
import tarfile

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--target-col", type=str, required=True)
    args = parser.parse_args()

    model_dir = "/opt/ml/processing/model"
    test_dir = "/opt/ml/processing/test"
    output_dir = "/opt/ml/processing/evaluation"
    os.makedirs(output_dir, exist_ok=True)

    tarball = os.path.join(model_dir, "model.tar.gz")
    if os.path.exists(tarball):
        with tarfile.open(tarball, "r:gz") as tar:
            tar.extractall(model_dir)

    bundle = joblib.load(os.path.join(model_dir, "model.joblib"))
    model = bundle["model"]
    feature_cols = bundle["feature_cols"]

    csv_files = [name for name in os.listdir(test_dir) if name.endswith(".csv")]
    test_df = pd.read_csv(os.path.join(test_dir, csv_files[0]))

    y_true = test_df[args.target_col]
    y_pred = model.predict(test_df[feature_cols])
    y_prob = model.predict_proba(test_df[feature_cols])[:, 1]

    metrics = {
        "test": {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred)),
            "recall": float(recall_score(y_true, y_pred)),
            "f1": float(f1_score(y_true, y_pred)),
            "roc_auc": float(roc_auc_score(y_true, y_prob)),
        }
    }
    print("Evaluation metrics:")
    print(json.dumps(metrics, indent=2))

    with open(os.path.join(output_dir, "evaluation.json"), "w") as f:
        json.dump(metrics, f, indent=2)


if __name__ == "__main__":
    main()
'''

eval_script_path = Path("scripts/evaluate.py")
eval_script_path.write_text(evaluation_script)
print(f"Wrote {eval_script_path}")

Wrote scripts/evaluate.py


## 4. Pipeline Parameters

`SampleSeed` and `SampleFrac` let scheduled runs train on a *different* subset
of the training data each fire, so daily executions produce slightly varied
metrics instead of identical results.

- `SampleSeed = 0`  → use full training set (default for manual runs, deterministic)
- `SampleSeed = -1` → derive a fresh seed from current time (set by the scheduler)
- `SampleSeed > 0`  → reproducible sample with that specific seed

In [20]:
min_f1_threshold = ParameterFloat(name="MinF1Threshold", default_value=0.85)
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.large")
sample_seed = ParameterInteger(name="SampleSeed", default_value=0)
sample_frac = ParameterFloat(name="SampleFrac", default_value=1.0)

print("Parameters defined.")

Parameters defined.


## 5. Training Step

Reuses `scripts/train_sentiment.py` from notebook 06 — no model code duplication.

In [21]:
sklearn_estimator = SKLearn(
    entry_point="scripts/train_sentiment.py",
    role=role,
    instance_type=training_instance_type,
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sm_session,
    hyperparameters={
        "feature-cols": ",".join(FEATURE_COLS),
        "target-col": TARGET_COL,
        "n-estimators": 100,
        "max-depth": 0,
        "random-state": RANDOM_STATE,
        "sample-seed": sample_seed,
        "sample-frac": sample_frac,
    },
)

train_step = TrainingStep(
    name="TrainRandomForest",
    estimator=sklearn_estimator,
    inputs={
        "train": TrainingInput(train_csv_s3, content_type="text/csv"),
        "validation": TrainingInput(validation_csv_s3, content_type="text/csv"),
        "test": TrainingInput(test_csv_s3, content_type="text/csv"),
    },
)
print("TrainingStep defined.")

TrainingStep defined.


## 6. Evaluation Step

ProcessingStep that runs `scripts/evaluate.py` and emits `evaluation.json` as a PropertyFile.

In [22]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    sagemaker_session=sm_session,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

eval_step = ProcessingStep(
    name="EvaluateModel",
    processor=sklearn_processor,
    code="scripts/evaluate.py",
    inputs=[
        ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=test_csv_s3,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
        ),
    ],
    property_files=[evaluation_report],
    job_arguments=["--target-col", TARGET_COL],
)
print("ProcessingStep defined.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


ProcessingStep defined.


## 7. Register Model Step

Runs only if the ConditionStep passes.

In [23]:
register_step = RegisterModel(
    name="RegisterModel",
    estimator=sklearn_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="yelp-sentiment-model-group",
    approval_status="PendingManualApproval",
)
print("RegisterModel step defined.")

RegisterModel step defined.


## 8. Fail Step

Runs only if the ConditionStep fails. Terminates the pipeline in Failed state.

In [24]:
fail_step = FailStep(
    name="ModelQualityBelowThreshold",
    error_message="Test F1 score did not meet MinF1Threshold parameter.",
)
print("FailStep defined.")

FailStep defined.


## 9. Condition Step

Reads `test.f1` out of the evaluation property file and branches to either
RegisterModel (if ≥ threshold) or the FailStep.

In [25]:
condition = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=eval_step.name,
        property_file=evaluation_report,
        json_path="test.f1",
    ),
    right=min_f1_threshold,
)

cond_step = ConditionStep(
    name="CheckF1Threshold",
    conditions=[condition],
    if_steps=[register_step],
    else_steps=[fail_step],
)
print("ConditionStep defined.")

ConditionStep defined.


## 10. Assemble & Upsert Pipeline

`upsert` is idempotent — creates the pipeline on first run, updates it on
subsequent runs. After this cell completes, the pipeline lives in AWS as a
managed resource (visible in SageMaker Studio → Pipelines) independent of
this notebook.

In [26]:
pipeline_name = "yelp-sentiment-pipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[min_f1_threshold, training_instance_type, sample_seed, sample_frac],
    steps=[train_step, eval_step, cond_step],
    sagemaker_session=sm_session,
)

pipeline.upsert(role_arn=role, tags=TAGS)

print(f"Pipeline upserted: {pipeline_name}")
print(f"Definition size: {len(pipeline.definition())} chars")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline upserted: yelp-sentiment-pipeline


Definition size: 5770 chars


## 11. Run Pipeline — Success Path

Default `MinF1Threshold=0.85`, `SampleSeed=0` (no sampling). The RF model
clears ~0.94 test F1, so the ConditionStep takes the "if" branch and
RegisterModel runs. Takes ~6–10 min.

In [27]:
success_execution = pipeline.start(
    execution_display_name=f"success-{run_id}",
)
print(f"Started: {success_execution.arn}")
print("Waiting (~6–10 min)...")
success_execution.wait(max_attempts=120, delay=30)
print(f"Final status: {success_execution.describe()['PipelineExecutionStatus']}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Started: arn:aws:sagemaker:us-east-1:476629097825:pipeline/yelp-sentiment-pipeline/execution/7nl4or9ibhov
Waiting (~6–10 min)...


Final status: Succeeded


## 12. Run Pipeline — Failure Path

Override `MinF1Threshold=0.99` (unreachable). ConditionStep takes the "else"
branch → FailStep. Pipeline ends in Failed state.

In [28]:
fail_execution = pipeline.start(
    parameters={"MinF1Threshold": 0.99},
    execution_display_name=f"fail-demo-{run_id}",
)
print(f"Started: {fail_execution.arn}")
print("Waiting (~6–10 min)...")
try:
    fail_execution.wait(max_attempts=120, delay=30)
except Exception as wait_error:
    # wait() raises when the pipeline ends in Failed state — that's the demo.
    print(f"Wait completed with: {type(wait_error).__name__}")

final_status = fail_execution.describe()["PipelineExecutionStatus"]
print(f"Final status: {final_status}")
assert final_status == "Failed", (
    f"Expected Failed, got {final_status}. Did the model exceed F1=0.99? "
    "Raise the threshold higher to ensure the FailStep fires."
)
print("Pipeline failed as expected.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Started: arn:aws:sagemaker:us-east-1:476629097825:pipeline/yelp-sentiment-pipeline/execution/18offgr3qtoh
Waiting (~6–10 min)...


Wait completed with: WaiterError
Final status: Failed
Pipeline failed as expected.


## 13. Describe Pipeline + Executions

Pipeline-level metadata + per-execution step states.

In [29]:
pipeline_description = sagemaker_client.describe_pipeline(PipelineName=pipeline_name)
print("Pipeline ARN:", pipeline_description["PipelineArn"])
print("Pipeline status:", pipeline_description["PipelineStatus"])
print("Created:", pipeline_description["CreationTime"])

Pipeline ARN: arn:aws:sagemaker:us-east-1:476629097825:pipeline/yelp-sentiment-pipeline
Pipeline status: Active
Created: 2026-05-29 03:07:41.266000+00:00


In [30]:
for label, execution in [("SUCCESS", success_execution), ("FAILURE", fail_execution)]:
    desc = execution.describe()
    print(f"\n=== {label} EXECUTION ===")
    print(f"ARN: {desc['PipelineExecutionArn']}")
    print(f"Status: {desc['PipelineExecutionStatus']}")
    print(f"Display name: {desc.get('PipelineExecutionDisplayName')}")
    if desc.get("FailureReason"):
        print(f"Failure reason: {desc['FailureReason']}")
    steps = sagemaker_client.list_pipeline_execution_steps(
        PipelineExecutionArn=desc["PipelineExecutionArn"]
    )["PipelineExecutionSteps"]
    print("Steps:")
    for step in steps:
        print(f"  - {step['StepName']:35s} {step['StepStatus']}")


=== SUCCESS EXECUTION ===
ARN: arn:aws:sagemaker:us-east-1:476629097825:pipeline/yelp-sentiment-pipeline/execution/7nl4or9ibhov
Status: Succeeded
Display name: success-2026-05-29-03-50-11
Steps:
  - RegisterModel-RegisterModel         Succeeded
  - CheckF1Threshold                    Succeeded
  - EvaluateModel                       Succeeded
  - TrainRandomForest                   Succeeded



=== FAILURE EXECUTION ===
ARN: arn:aws:sagemaker:us-east-1:476629097825:pipeline/yelp-sentiment-pipeline/execution/18offgr3qtoh
Status: Failed
Display name: fail-demo-2026-05-29-03-50-11
Failure reason: Step failure: One or multiple steps failed.


Steps:
  - ModelQualityBelowThreshold          Failed
  - CheckF1Threshold                    Succeeded
  - EvaluateModel                       Succeeded
  - TrainRandomForest                   Succeeded


## 14. Schedule Daily Pipeline Executions (EventBridge Scheduler)

Creates a recurring schedule that fires the pipeline every 24 hours. Each
scheduled execution passes `SampleSeed=-1`, which makes the training script
derive a fresh seed from epoch time — so each daily run trains on a different
80% subset and produces meaningfully varied metrics.

**IAM requirement:** the role used here must be assumable by
`scheduler.amazonaws.com`. AWS Academy `LabRole` typically satisfies this.
If it doesn't, the create_schedule call will return an access-denied error;
in that case you'll need to use a role with the right trust policy.

In [32]:
scheduler = session.client("scheduler")

schedule_name = "yelp-pipeline-daily"

# ClientRequestToken is required by SageMaker's StartPipelineExecution API.
# boto3 auto-generates one for direct calls, but EventBridge Scheduler's
# universal SDK target does not — we must include it in the Input payload.
# The <aws.scheduler.execution-id> placeholder resolves to a unique UUID per
# scheduled invocation, so every daily fire gets a distinct idempotency token.
schedule_input = json.dumps({
    "PipelineName": pipeline_name,
    "PipelineExecutionDisplayName": "scheduled-run",
    "ClientRequestToken": "<aws.scheduler.execution-id>",
    "PipelineParameters": [
        {"Name": "MinF1Threshold", "Value": "0.85"},
        {"Name": "SampleSeed", "Value": "-1"},
        {"Name": "SampleFrac", "Value": "0.8"},
    ],
})

schedule_target = {
    "Arn": "arn:aws:scheduler:::aws-sdk:sagemaker:startPipelineExecution",
    "RoleArn": role,
    "Input": schedule_input,
}

try:
    scheduler.create_schedule(
        Name=schedule_name,
        ScheduleExpression="rate(24 hours)",
        FlexibleTimeWindow={"Mode": "OFF"},
        Target=schedule_target,
        Description="Daily Yelp sentiment pipeline run with 80% random subsample.",
    )
    print(f"Created schedule: {schedule_name}")
except scheduler.exceptions.ConflictException:
    scheduler.update_schedule(
        Name=schedule_name,
        ScheduleExpression="rate(24 hours)",
        FlexibleTimeWindow={"Mode": "OFF"},
        Target=schedule_target,
    )
    print(f"Updated existing schedule: {schedule_name}")

description = scheduler.get_schedule(Name=schedule_name)
print(f"Schedule ARN: {description['Arn']}")
print(f"State: {description['State']}")
print(f"Next fire: see EventBridge Scheduler console")

Created schedule: yelp-pipeline-daily
Schedule ARN: arn:aws:scheduler:us-east-1:476629097825:schedule/default/yelp-pipeline-daily
State: ENABLED
Next fire: see EventBridge Scheduler console


## 15. Inspect Scheduled Executions

Run this any time over the next few days to see what the scheduler has
triggered. Compares F1 scores across executions to surface variation.

In [33]:
executions = sagemaker_client.list_pipeline_executions(
    PipelineName=pipeline_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=20,
)["PipelineExecutionSummaries"]

print(f"Last {len(executions)} executions of {pipeline_name}:")
for e in executions:
    print(f"  {e['StartTime'].isoformat():25s}  "
          f"{e.get('PipelineExecutionDisplayName','?'):30s}  "
          f"{e['PipelineExecutionStatus']}")

Last 4 executions of yelp-sentiment-pipeline:
  2026-05-29T03:55:46.554000+00:00  fail-demo-2026-05-29-03-50-11   Failed
  2026-05-29T03:50:14.957000+00:00  success-2026-05-29-03-50-11     Succeeded
  2026-05-29T03:15:43.941000+00:00  fail-demo-2026-05-29-03-07-38   Failed
  2026-05-29T03:07:42.038000+00:00  success-2026-05-29-03-07-38     Succeeded


## 16. Stop the Schedule

When you're done monitoring, run this cell to delete the schedule. Existing
pipeline executions stay in history; only the recurring trigger is removed.

In [ ]:
scheduler.delete_schedule(Name=schedule_name)
print(f"Deleted schedule: {schedule_name}")